In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
event_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("event_type", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("city", StringType(), True)
])

In [0]:
silver_df = spark.readStream.table("ecommerce.bronze.bronze_events")
silver_df = (
    silver_df.dropDuplicates(["user_id"]) \
)
silver_df = silver_df.withColumn("amount",col("amount").cast("double"))
# display(silver_stream,checkpointLocation="/Volumes/file_upload/files/tmp/delta/bronze_event_checkpoint/")

In [0]:
silver_df = silver_df.dropna(subset=["event_id","event_type","user_id","product_id","qty","amount","timestamp","city"])
silver_df = silver_df.fillna({"city" : "Unknown","amount" : 0})

In [0]:
silver_df = (
    silver_df.withColumn("event_id",trim(col("event_id"))) \
    .withColumn("event_type",trim(col("event_type"))) \
        .withColumn("user_id",trim(col("user_id"))) \
            .withColumn("product_id",trim(col("product_id"))) \
                .withColumn("city",trim(col("city")))
)

In [0]:
silver_df = silver_df.filter(col("amount") > 0) \
    .filter(col("qty") > 0)

In [0]:
fraud_df = (
    silver_df
    .withWatermark("timestamp","1 minute")
    .groupBy(
        window(col("timestamp"),"20 seconds"),
        col("user_id")
    )
    .agg(
        count("*").alias("transcation_count")
    )
    .withColumn("fraud_flag",(col("transcation_count") >= 3).cast("int"))
)

In [0]:
(silver_df.writeStream
    .outputMode("append")
    .option("checkpointLocation","/Volumes/file_upload/files/tmp/delta/silver_event_checkpoint/")
    .trigger(availableNow=True) 
    .toTable("ecommerce.silver.silver_events"))